## Key CNN Concepts Summary

### 1. Convolution
- Sliding a filter over the image to extract features
- Detects edges, textures, shapes in early layers
- More complex patterns in deeper layers
- Output shape: (Height - Filter + 1) × (Width - Filter + 1)

### 2. Pooling (Max Pooling)
- Downsampling to reduce parameters and computation
- Keeps the most important feature in each window
- Reduces overfitting by discarding sensitive features
- Output shape: (Height / PoolSize) × (Width / PoolSize)

### 3. ReLU Activation
- Introduces non-linearity: max(0, x)
- Prevents gradient vanishing problem
- Standard choice for hidden layers

### 4. Flattening
- Converts 2D/3D feature maps to 1D vector
- Bridge between convolutional layers and dense layers

### 5. Dropout
- Randomly deactivates neurons during training
- Forces network to learn redundant representations
- Reduces co-adaptation of neurons
- Prevents overfitting

### 6. Softmax Activation
- Multi-class classification activation
- Converts raw scores to probability distribution
- Sum of all outputs = 1
- Used in final output layer

### Performance on MNIST
- Expected Accuracy: **97-99%**
- CNN learns spatial hierarchies of features automatically
- Much more efficient than ANN for image data

In [ ]:
# Get predictions on test set
predictions = model.predict(X_test[:10])

# Display predictions vs actual
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_test[i].reshape(28, 28), cmap='gray')
    pred = np.argmax(predictions[i])
    actual = np.argmax(y_test[i])
    color = 'green' if pred == actual else 'red'
    ax.set_title(f"Pred: {pred}, Actual: {actual}", color=color)
    ax.axis('off')
plt.tight_layout()
plt.show()

# Show confidence scores
print("\nPrediction Confidence Scores for first 5 test images:")
for i in range(5):
    pred = np.argmax(predictions[i])
    confidence = predictions[i][pred] * 100
    actual = np.argmax(y_test[i])
    print(f"Image {i}: Predicted={pred}, Confidence={confidence:.2f}%, Actual={actual}")

## Section 8: Make Predictions

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.4f}")

# Plot accuracy and loss
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Accuracy plot
axes[0].plot(history.history['accuracy'], label='Training')
axes[0].plot(history.history['val_accuracy'], label='Validation')
axes[0].set_title('Model Accuracy')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True)

# Loss plot
axes[1].plot(history.history['loss'], label='Training')
axes[1].plot(history.history['val_loss'], label='Validation')
axes[1].set_title('Model Loss')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

## Section 7: Evaluate the Model

In [ ]:
history = model.fit(
    X_train, y_train,
    epochs=15,
    batch_size=128,
    validation_split=0.2,  # 20% for validation
    callbacks=[early_stop, tensorboard_callback]
)

## Section 6: Train the Model

In [ ]:
model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',  # Multi-class classification
    metrics=['accuracy']
)

# Setup Early Stopping
early_stop = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

# Setup TensorBoard
log_dir = "cnn_logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

## Section 5: Compile the Model

In [ ]:
model = Sequential([
    # First Convolutional Block
    Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    # Filters: 32 | Kernel size: 3x3 | Activation: ReLU
    MaxPooling2D((2, 2)),
    # Reduces spatial dimensions by half
    
    # Second Convolutional Block
    Conv2D(64, (3, 3), activation='relu'),
    # Filters: 64 | Detects more complex features
    MaxPooling2D((2, 2)),
    
    # Third Convolutional Block
    Conv2D(128, (3, 3), activation='relu'),
    
    # Flatten to 1D for Dense layers
    Flatten(),
    
    # Dense layers for classification
    Dense(128, activation='relu'),
    Dropout(0.5),  # Prevent overfitting
    Dense(10, activation='softmax')  # 10 output classes
])

model.summary()

## Section 4: Build the CNN Model

In [ ]:
# Normalize pixel values to 0-1 range
X_train = X_train.astype('float32') / 255.0
X_test = X_test.astype('float32') / 255.0

# Reshape data: (samples, height, width, channels)
# MNIST is grayscale, so 1 channel
X_train = X_train.reshape(-1, 28, 28, 1)
X_test = X_test.reshape(-1, 28, 28, 1)

print(f"Reshaped training data: {X_train.shape}")
print(f"Reshaped test data: {X_test.shape}")

# Convert labels to one-hot encoding (0-9 → 10 classes)
y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

print(f"One-hot encoded training labels shape: {y_train.shape}")

## Section 3: Preprocess the Data

In [ ]:
# Display first 9 images
fig, axes = plt.subplots(3, 3, figsize=(8, 8))
for i, ax in enumerate(axes.flat):
    ax.imshow(X_train[i], cmap='gray')
    ax.set_title(f"Label: {y_train[i]}")
    ax.axis('off')
plt.tight_layout()
plt.show()

### Visualize Sample Images

In [ ]:
# Load MNIST dataset (70,000 images of handwritten digits 0-9)
(X_train, y_train), (X_test, y_test) = mnist.load_data()

print(f"Training data shape: {X_train.shape}")  # (60000, 28, 28)
print(f"Test data shape: {X_test.shape}")        # (10000, 28, 28)
print(f"Training labels shape: {y_train.shape}")
print(f"Pixel values range: {X_train.min()} to {X_train.max()}")
print(f"Unique classes: {np.unique(y_train)}")

## Section 2: Load and Explore MNIST Dataset

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Dropout
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
from tensorflow.keras.utils import to_categorical
import numpy as np
import matplotlib.pyplot as plt
import datetime

print(f"TensorFlow version: {tf.__version__}")

## Section 1: Import Required Libraries

# Convolutional Neural Networks (CNN) on MNIST Dataset

## Introduction
Convolutional Neural Networks use convolutional layers to automatically learn spatial features from images.
Key components: **Convolution → Activation → Pooling → Flattening → Dense layers**

This notebook will build and train a CNN model to classify handwritten digits (0-9) from the MNIST dataset.